In [9]:
!pip install torch==1.13.1
!pip install torchtext==0.14.1
!pip install torchdata==0.5.1

import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# 设置随机种子以确保结果可复现
np.random.seed(7)
tf.random.set_seed(7)

# 超参数
max_features = 10000  # 仅考虑最常见的10000个单词
maxlen = 500          # 每条评论的最大长度
batch_size = 32       # 批大小

# 加载IMDb数据集
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

# 填充序列，使它们具有相同的长度
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)

# 构建模型
model = Sequential()
model.add(Embedding(max_features, 32, input_length=maxlen))
model.add(SimpleRNN(32))
model.add(Dense(1, activation='sigmoid'))

# 编译模型
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 显示模型摘要
model.summary()

# 训练模型
history = model.fit(x_train, y_train, epochs=10, batch_size=batch_size, validation_split=0.2)

# 计算困惑度的函数
def perplexity(y_true, y_pred):
    cross_entropy = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.exp(cross_entropy)

# 在测试集上评估模型
y_pred = model.predict(x_test, batch_size=batch_size)
perplex = perplexity(y_test, y_pred).numpy()

print(f'困惑度: {np.mean(perplex)}')

17464789/17464789 [==============================] - 0s 0us/step
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 500, 32)           320000    
                                                                 
 simple_rnn (SimpleRNN)      (None, 32)                2080      
                                                                 
 dense (Dense)               (None, 1)                 33        
                                                                 
Total params: 322113 (1.23 MB)
Trainable params: 322113 (1.23 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/10
625/625 [==============================] - 104s 164ms/step - loss: 0.6108 - accuracy: 0.6538 - val_loss: 0.5095 - val_accuracy: 0.7570
Epoch 2/10
625/625 [==============================] - 84s 135ms/step - lo